# KIKA: Sensitivity Comparison and UQ Pipeline

This notebook demonstrates how to compare sensitivities from different sources (MCNP, Serpent, or any SDF file) and perform unified uncertainty quantification.

The `ComparisonReport` class enables source-agnostic comparison via the SDFData common format.

# KIKA: Sensitivity Comparison and UQ Pipeline

This notebook demonstrates how to compare sensitivities from different sources (MCNP, Serpent, or any SDF file) and perform unified uncertainty quantification.

## Use Cases

1. **Code-to-code validation**: Compare MCNP vs Serpent sensitivities
2. **Benchmark analysis**: Compare calculated vs reference sensitivities
3. **Method validation**: Compare different perturbation approaches
4. **Multi-source UQ**: Combine sensitivities from multiple calculations

## Architecture

```
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│   MCNP PERT     │     │   Serpent       │     │   SDF File      │
│   Results       │     │   .sens File    │     │   (any source)  │
└────────┬────────┘     └────────┬────────┘     └────────┬────────┘
         │                       │                       │
         ▼                       ▼                       ▼
    ┌─────────┐            ┌─────────┐            ┌─────────┐
    │ SDFData │            │ SDFData │            │ SDFData │
    └────┬────┘            └────┬────┘            └────┬────┘
         │                      │                      │
         └──────────────────────┼──────────────────────┘
                                │
                                ▼
                      ┌─────────────────────┐
                      │  ComparisonReport   │
                      │  - Comparison tables │
                      │  - Difference plots  │
                      │  - Scatter diagrams  │
                      └─────────────────────┘
```

**Key Insight**: SDFData is the common interchange format that enables source-agnostic comparison.

In [1]:
import kika
from kika.sensitivities import (
    SDFData,
    ComparisonReport,
    UQReport,
    compute_sensitivity,
    create_sdf_data
)
from kika.sensitivities.sdf_parser import read_sdf
from pathlib import Path
import pandas as pd
import numpy as np

# Setup paths
repo_root = Path.cwd().resolve().parent
examples_dir = repo_root / 'examples'
output_dir = Path.cwd() / 'output'
output_dir.mkdir(exist_ok=True)

print(f"Examples directory: {examples_dir}")
print(f"Output directory: {output_dir}")

Examples directory: /home/MONLEON-JUAN/kika/examples
Output directory: /home/MONLEON-JUAN/kika/examples/output


## Step 1: Load/Create SDFData from Different Sources

We'll demonstrate three ways to get SDFData:
1. From MCNP PERT results
2. From Serpent sensitivity files
3. From existing SDF files

### 1.1 Create SDF from MCNP PERT

In [2]:
# Load MCNP PERT data
data_dir = examples_dir / 'data'
inputfile = data_dir / 'inputfile_example_1.i'
mctalfile = data_dir / 'mctalfile_example_1.m'

if inputfile.exists() and mctalfile.exists():
    # Compute sensitivity
    sens_fe56 = compute_sensitivity(
        inputfile=str(inputfile),
        mctalfile=str(mctalfile),
        tally=4,
        zaid=26056,
        label="MCNP Fe-56"
    )
    
    # Convert to SDF
    sdf_mcnp = create_sdf_data(
        sens_list=[sens_fe56],
        energy='integral',
        title='MCNP_PERT'
    )
    
    print(f"MCNP SDF: {len(sdf_mcnp.data)} reactions")
else:
    sdf_mcnp = None
    print("MCNP example files not found")

MCNP SDF: 8 reactions


### 1.2 Create SDF from Serpent

In [3]:
from kika.serpent.parse_sens import read_sensitivity_file
from kika.sensitivities.sensitivity_processing import create_sdf_from_serpent

# Path to Serpent file
serpent_file = repo_root / 'kika' / 'serpent' / 'PWRSphere.sss2_sens0.m'

if serpent_file.exists():
    sf = read_sensitivity_file(str(serpent_file))
    
    sdf_serpent = create_sdf_from_serpent(
        serpent_file=sf,
        response_name=sf.responses[0],
        title='Serpent_Adjoint'
    )
    
    print(f"Serpent SDF: {len(sdf_serpent.data)} reactions")
else:
    sdf_serpent = None
    print("Serpent example file not found")

Processing file 1/1 with response 'sens_ratio_BIN_0'...
Combined SDF contains 6 sensitivity profiles from 1 files
Serpent SDF: 6 reactions


### 1.3 Load Existing SDF File

In [4]:
# Check for existing SDF files
sdf_files = list(examples_dir.glob('*.sdf'))
print(f"Found {len(sdf_files)} SDF files in examples/")

sdf_loaded = None
if sdf_files:
    # Load the first one
    sdf_file = sdf_files[0]
    print(f"\nLoading: {sdf_file.name}")
    
    sdf_loaded = read_sdf(str(sdf_file))
    print(f"Loaded SDF: {len(sdf_loaded.data)} reactions")
    print(sdf_loaded)

Found 2 SDF files in examples/

Loading: Example_SDF_Dataset_1.00e+00_3.00e+00.sdf


ValueError: Malformed reaction header at line 15: H-1          total             1001      1

## Step 2: Create Synthetic Data for Demo

If real data is not available, let's create synthetic SDFData objects for demonstration.

In [ ]:
from kika.sensitivities.sdf import SDFData, SDFReactionData

def create_synthetic_sdf(title, scale_factor=1.0, noise_level=0.1):
    """Create synthetic SDF data for demonstration."""
    # Define energy grid (10 groups)
    energies = np.logspace(-5, 1, 11)  # 1e-5 to 10 MeV
    
    # Create reactions for Fe-56
    reactions = []
    
    # MT 2: Elastic
    sens_elastic = np.array([0.05, 0.08, 0.12, 0.15, 0.10, 0.07, 0.04, 0.02, 0.01, 0.005])
    sens_elastic = sens_elastic * scale_factor * (1 + noise_level * np.random.randn(len(sens_elastic)))
    reactions.append(SDFReactionData(
        zaid=26056,
        mt=2,
        sensitivity=sens_elastic.tolist(),
        error=[0.05] * len(sens_elastic)
    ))
    
    # MT 102: Capture
    sens_capture = np.array([0.02, 0.03, 0.05, 0.08, 0.12, 0.15, 0.10, 0.05, 0.02, 0.01])
    sens_capture = sens_capture * scale_factor * (1 + noise_level * np.random.randn(len(sens_capture)))
    reactions.append(SDFReactionData(
        zaid=26056,
        mt=102,
        sensitivity=sens_capture.tolist(),
        error=[0.08] * len(sens_capture)
    ))
    
    # MT 4: Inelastic
    sens_inelastic = np.array([0.0, 0.0, 0.01, 0.03, 0.06, 0.08, 0.10, 0.08, 0.05, 0.02])
    sens_inelastic = sens_inelastic * scale_factor * (1 + noise_level * np.random.randn(len(sens_inelastic)))
    reactions.append(SDFReactionData(
        zaid=26056,
        mt=4,
        sensitivity=sens_inelastic.tolist(),
        error=[0.10] * len(sens_inelastic)
    ))
    
    return SDFData(
        title=title,
        energy='1.00e-05_1.00e+01',
        pert_energies=energies.tolist(),
        r0=1.0,
        e0=0.01,
        data=reactions
    )

# Create synthetic datasets
np.random.seed(42)  # For reproducibility
sdf_code_a = create_synthetic_sdf('Code_A_Reference', scale_factor=1.0, noise_level=0.0)
sdf_code_b = create_synthetic_sdf('Code_B_Test', scale_factor=0.95, noise_level=0.05)

print(f"Created synthetic SDFs:")
print(f"  - {sdf_code_a.title}: {len(sdf_code_a.data)} reactions")
print(f"  - {sdf_code_b.title}: {len(sdf_code_b.data)} reactions")

## Step 3: Sensitivity Comparison

The `ComparisonReport` class compares multiple SDFData objects and generates:
- Common reaction identification
- Comparison tables
- Difference analysis (absolute and relative)
- Visualization (bar charts, scatter plots)

In [ ]:
# Create comparison report
comparison = ComparisonReport(
    sdf_list=[sdf_code_a, sdf_code_b],
    labels=['Reference', 'Test Code'],
    title='Sensitivity Comparison Demo'
)

# Find common reactions
common = comparison.find_common_reactions()
print(f"Common reactions found: {len(common)}")
for zaid, mt in common:
    from kika._constants import ATOMIC_NUMBER_TO_SYMBOL, MT_TO_REACTION
    z = zaid // 1000
    a = zaid % 1000
    nuclide = f"{ATOMIC_NUMBER_TO_SYMBOL.get(z, f'Z{z}')}-{a}"
    reaction = MT_TO_REACTION.get(mt, f'MT{mt}')
    print(f"  {nuclide} {reaction} (ZAID={zaid}, MT={mt})")

In [ ]:
# Comparison table
comp_table = comparison.comparison_table()
comp_table

In [ ]:
# Difference analysis (relative to first dataset)
diff_table = comparison.difference_table(reference_idx=0)

# Show only difference columns
diff_cols = ['Nuclide', 'Reaction'] + [c for c in diff_table.columns if 'vs' in c or '-' in c]
diff_table[diff_cols]

### Comparison Visualizations

In [ ]:
# Bar chart comparison
fig = comparison.plot_comparison(top_n=6)
fig

In [ ]:
# Scatter plot (X vs Y)
fig = comparison.plot_scatter(x_idx=0, y_idx=1)
fig

### Save Comparison Report

In [ ]:
# Save HTML report
report_path = output_dir / 'sensitivity_comparison.html'
comparison.save_html(str(report_path), include_plots=True)
print(f"Report saved: {report_path}")

## Step 4: Compare Real Data (if available)

If we have both MCNP and Serpent data, let's compare them.

In [ ]:
# Check if we have real data to compare
real_data = []
labels = []

if sdf_mcnp is not None:
    real_data.append(sdf_mcnp)
    labels.append('MCNP PERT')
    
if sdf_serpent is not None:
    real_data.append(sdf_serpent)
    labels.append('Serpent')
    
if sdf_loaded is not None:
    real_data.append(sdf_loaded)
    labels.append('SDF File')

if len(real_data) >= 2:
    print(f"Comparing {len(real_data)} real datasets: {labels}")
    
    real_comparison = ComparisonReport(
        sdf_list=real_data,
        labels=labels,
        title='Cross-Code Sensitivity Comparison'
    )
    
    # Show common reactions
    common = real_comparison.find_common_reactions()
    print(f"\nCommon reactions: {len(common)}")
    
    if common:
        # Comparison table
        display(real_comparison.comparison_table())
        
        # Plot
        fig = real_comparison.plot_comparison()
else:
    print(f"Only {len(real_data)} dataset(s) available - need at least 2 for comparison")

## Step 5: Unified UQ from SDFData

Once you have SDFData (from any source), you can run uncertainty quantification using the same interface.

In [ ]:
# UQ workflow is identical regardless of sensitivity source
#
# from kika.cov import CovMat
#
# # Pick any SDF source
# sdf = sdf_mcnp  # or sdf_serpent, or sdf_loaded
#
# # Load covariance matrices
# cov = CovMat.from_file('path/to/covmat.dat')
#
# # Run UQ
# uq = UQReport(
#     sdf_data=sdf,
#     cov_mat=cov,
#     title='Unified UQ'
# )
# uq.compute()
#
# # Results
# print(uq.result)
# uq.save_html('uq_report.html')

print("UQ pipeline is source-agnostic once data is in SDFData format.")
print("")
print("Benefits of unified SDFData approach:")
print("  1. Same UQ code for MCNP, Serpent, SCALE, etc.")
print("  2. Easy comparison between codes")
print("  3. Standard file format for archival")
print("  4. Reproducible analysis pipelines")

## Summary

This notebook demonstrated the sensitivity comparison and unified UQ pipeline:

| Step | Input | Tool | Output |
|------|-------|------|--------|
| 1 | MCNP/Serpent/SDF | Various parsers | SDFData |
| 2 | Multiple SDFData | `ComparisonReport` | Tables, plots, HTML |
| 3 | Any SDFData + CovMat | `UQReport` | Uncertainty breakdown |

### Key Takeaways

1. **SDFData as common format**: All sources convert to SDFData
2. **Source-agnostic comparison**: `ComparisonReport` works with any SDFData
3. **Unified UQ**: Same sandwich formula for all sensitivity sources
4. **Extensible**: Easy to add new sensitivity sources

### Workflow Diagram

```
                    ┌────────────────────┐
                    │   ComparisonReport │
                    │   (Multi-source)   │
                    └─────────┬──────────┘
                              │
    ┌─────────────────────────┼─────────────────────────┐
    │                         │                         │
┌───┴───┐                 ┌───┴───┐                 ┌───┴───┐
│SDFData│                 │SDFData│                 │SDFData│
│ MCNP  │                 │Serpent│                 │ File  │
└───┬───┘                 └───┬───┘                 └───┬───┘
    │                         │                         │
    │                         │                         │
    └─────────────────────────┴─────────────────────────┘
                              │
                              ▼
                    ┌────────────────────┐
                    │     UQReport       │
                    │  (Any SDF source)  │
                    └────────────────────┘
```

In [ ]:
# Final summary of outputs
print("Generated outputs:")
for f in output_dir.glob('*'):
    print(f"  - {f.name}")